# HUDA — Flood-rain classifier (LightGBM)

Trains `lgbm_v1.txt` for the backend (architecture.md §5.2, implementation-plan.md §5.6).

**Leakage guard (D3):** lag/weather features come from the Open-Meteo **Archive** (what happened);
the two forecast-precipitation features come from the **Historical Forecast API** (what the model
forecast at the time). Never build forecast features from the Archive — its "future" rain *is* the label.

Features are computed with `server/app/services/features.py` — the same code the backend runs.

Run top to bottom in Colab. Downloads are cached in `DATA_DIR`, so re-runs are fast.

In [ ]:
!pip -q install lightgbm scikit-learn pandas matplotlib requests
import json, math, os, sys, time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import requests

REPO = Path(os.environ.get("HUDA_REPO", "/content/huda"))  # !git clone <your repo url> /content/huda
sys.path.insert(0, str(REPO / "server"))
from app.services.features import FEATURE_NAMES, HOURLY_VARS, build_features, label  # noqa: E402

DATA_DIR = Path(os.environ.get("HUDA_DATA", REPO / "ml" / "data"))
MODEL_DIR = REPO / "ml" / "models"
DATA_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

START, END = "2016-01-01", "2025-12-31"
TRAIN_END, VALID_END = "2022-12-31", "2023-12-31"  # train 2016–22 · validate 2023 · test 2024–25
LABEL_MM = 30.0  # proxy label: >= 30 mm in the next 3 h. Lower it if positives < ~0.3 % (step 4)

USE_FORECAST_FEATURES = True
# MUST differ from the Archive's source. From 2017 the Archive *is* the stitched ECMWF IFS forecast,
# so ecmwf_ifs / best_match historical forecasts are byte-identical to it (= the label). GFS is
# independent (r ≈ 0.25 with the Archive at Elmina) and archived from ~2021-03. The backend fetches the
# same model live (features.json "forecast_model").
FCST_MODEL = "gfs_seamless"
FCST_START = "2021-04-01"

In [ ]:
ZONES = [  # spec §5 (approximate — keep in sync with supabase/seed_zones.sql)
    ("city-of-elmina", 3.183, 101.486), ("denai-alam", 3.140, 101.495), ("bukit-jelutong", 3.105, 101.531),
    ("bukit-subang", 3.160, 101.520), ("ara-damansara", 3.117, 101.582), ("klgcc-resort", 3.155, 101.625),
    ("bandar-bukit-raja", 3.090, 101.430), ("putra-heights", 2.996, 101.573), ("usj-heights", 3.046, 101.574),
    ("serenia-city", 2.870, 101.690), ("saujana-impian", 2.990, 101.795), ("nilai-impian", 2.810, 101.770),
    ("bandar-ainsdale", 2.700, 101.930), ("bandar-universiti-pagoh", 2.150, 102.730),
    ("taman-pasir-putih", 1.470, 103.890),
]
ZONES = ZONES[: int(os.environ.get("HUDA_MAX_ZONES", len(ZONES)))]

## 1–2. Download (cached)

In [ ]:
ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"
HIST_FCST_URL = "https://historical-forecast-api.open-meteo.com/v1/forecast"


def fetch(url, params, cache_name, retries=4):
    path = DATA_DIR / cache_name
    if path.exists():
        return json.loads(path.read_text())
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, timeout=120)
        except (requests.ConnectionError, requests.Timeout):
            time.sleep(10 * (attempt + 1))
            continue
        if r.status_code == 429:  # free-tier minutely limit
            time.sleep(60)
            continue
        r.raise_for_status()
        path.write_text(r.text)
        return r.json()
    raise RuntimeError(f"download failed after {retries} tries: {cache_name}")


def to_series(data, variables):
    h = data["hourly"]
    s = {v: h.get(v, []) for v in variables}
    s["time"] = [datetime.fromisoformat(t).replace(tzinfo=timezone.utc) for t in h["time"]]
    return s


def years(start, end):
    for y in range(int(start[:4]), int(end[:4]) + 1):
        yield max(start, f"{y}-01-01"), min(end, f"{y}-12-31")


def download_zone(slug, lat, lon):
    obs = {v: [] for v in HOURLY_VARS} | {"time": []}
    for a, b in years(START, END):
        d = fetch(ARCHIVE_URL, {"latitude": lat, "longitude": lon, "start_date": a, "end_date": b,
                                "hourly": ",".join(HOURLY_VARS), "timezone": "GMT"}, f"archive_{slug}_{a[:4]}.json")
        s = to_series(d, HOURLY_VARS)
        for k in obs:
            obs[k] += s[k]
    fcst = None
    if USE_FORECAST_FEATURES:
        fcst = {"precipitation": [], "time": []}
        for a, b in years(max(START, FCST_START), END):
            d = fetch(HIST_FCST_URL, {"latitude": lat, "longitude": lon, "start_date": a, "end_date": b,
                                      "hourly": "precipitation", "models": FCST_MODEL, "timezone": "GMT"},
                      f"fcst_{FCST_MODEL}_{slug}_{a[:4]}.json")
            s = to_series(d, ["precipitation"])
            fcst["precipitation"] += s["precipitation"]
            fcst["time"] += s["time"]
    return obs, fcst


raw = {}
for slug, lat, lon in ZONES:
    raw[slug] = download_zone(slug, lat, lon)
    print(slug, len(raw[slug][0]["time"]), "hours")

## 3. Features + proxy label (shared feature code)

In [ ]:
def align(obs, fcst):
    """Re-index the forecast series onto the archive's hourly timeline (missing -> None)."""
    if not fcst:
        return None
    by_time = dict(zip(fcst["time"], fcst["precipitation"]))
    return {"precipitation": [by_time.get(t) for t in obs["time"]], "time": obs["time"]}


rows = []
for slug, lat, lon in ZONES:
    obs, fcst = raw[slug]
    fc = align(obs, fcst)
    times = obs["time"]
    for t in range(24, len(times) - 3):
        y = label(obs, t, LABEL_MM)
        if y is None:
            continue
        f = build_features(obs, t, lat, lon, times[t], fcst=fc)
        f.update(zone=slug, ts=times[t], y=y)
        rows.append(f)
df = pd.DataFrame(rows)
print(df.shape)

## 4. Check the positive rate *before* training
Archive data is a coarse reanalysis grid that smooths convective peaks, so ≥ 30 mm / 3 h may be far
rarer than the 1–3 % architecture.md expects. If it is below ~0.3 %, lower `LABEL_MM` (e.g. 20) and
record the change in architecture.md §5.2.

In [ ]:
# Leakage check (D3): the forecast series must not be the Archive series.
if USE_FORECAST_FEATURES:
    obs0, fc0 = raw[ZONES[0][0]][0], align(*raw[ZONES[0][0]])
    pairs = [(a, b) for a, b in zip(obs0["precipitation"], fc0["precipitation"]) if a is not None and b is not None]
    same = np.mean([a == b for a, b in pairs])
    print(f"forecast == archive on {same:.1%} of hours")
    assert same < 0.8, "forecast features are the Archive series -> label leakage; change FCST_MODEL"

pos_rate = df["y"].mean()
print(f"positive rate at {LABEL_MM:g} mm/3 h: {pos_rate:.3%}  ({int(df['y'].sum())} positives)")
obs0 = raw[ZONES[0][0]][0]  # quick sensitivity check on one zone
for mm in (15, 20, 25, 30):
    labels = [label(obs0, t, mm) for t in range(24, len(obs0["time"]) - 3)]
    print(f"  {mm} mm/3 h -> {np.mean([y or 0 for y in labels]):.3%} ({ZONES[0][0]})")
if USE_FORECAST_FEATURES:
    print("forecast-feature coverage:", f"{df['fcst_precip_3h'].notna().mean():.1%}")

## 5–6. Time-based split + LightGBM
Several Klang Valley zones share one reanalysis grid cell, so rows are near-duplicates across zones.
Split by time only — a random split would leak.

In [ ]:
import lightgbm as lgb  # noqa: E402
from sklearn.metrics import average_precision_score, brier_score_loss, precision_recall_curve, roc_auc_score  # noqa: E402

features = [f for f in FEATURE_NAMES if USE_FORECAST_FEATURES or not f.startswith("fcst_")]
if USE_FORECAST_FEATURES:
    df = df[df["ts"] >= pd.Timestamp(FCST_START, tz="UTC")]  # train only where forecast features exist
train = df[df["ts"] <= pd.Timestamp(TRAIN_END + " 23:00", tz="UTC")]
valid = df[(df["ts"] > pd.Timestamp(TRAIN_END + " 23:00", tz="UTC")) & (df["ts"] <= pd.Timestamp(VALID_END + " 23:00", tz="UTC"))]
test = df[df["ts"] > pd.Timestamp(VALID_END + " 23:00", tz="UTC")]
print({k: (len(v), int(v["y"].sum())) for k, v in {"train": train, "valid": valid, "test": test}.items()})

spw = (len(train) - train["y"].sum()) / max(train["y"].sum(), 1)
params = {"objective": "binary", "learning_rate": 0.03, "num_leaves": 31, "min_data_in_leaf": 200,
          "feature_fraction": 0.8, "bagging_fraction": 0.8, "bagging_freq": 1, "lambda_l2": 1.0,
          "scale_pos_weight": spw, "metric": "average_precision", "verbose": -1}
dtrain = lgb.Dataset(train[features], train["y"])
dvalid = lgb.Dataset(valid[features], valid["y"], reference=dtrain)
model = lgb.train(params, dtrain, num_boost_round=3000, valid_sets=[dvalid],
                  callbacks=[lgb.early_stopping(150), lgb.log_evaluation(200)])

## 7. Metrics + AMBER/RED probability cut-offs
Note: `scale_pos_weight` inflates raw probabilities, so the cut-offs are chosen on validation —
don't read the scores as calibrated probabilities. The Brier score is reported for completeness.

In [ ]:
def evaluate(part, name):
    p = model.predict(part[features], num_iteration=model.best_iteration)
    m = {"roc_auc": roc_auc_score(part["y"], p), "pr_auc": average_precision_score(part["y"], p),
         "brier": brier_score_loss(part["y"], p), "positives": int(part["y"].sum()), "rows": len(part)}
    print(name, {k: round(v, 4) if isinstance(v, float) else v for k, v in m.items()})
    return p, m


p_valid, m_valid = evaluate(valid, "valid")
p_test, m_test = evaluate(test, "test")

prec, rec, thr = precision_recall_curve(valid["y"], p_valid)
prec, rec = prec[:-1], rec[:-1]
red_idx = np.where(prec >= 0.5)[0]
red_p = float(thr[red_idx[0]]) if len(red_idx) else float(np.quantile(p_valid, 0.995))
amber_idx = np.where((rec >= 0.8) & (thr < red_p))[0]
amber_p = float(thr[amber_idx[-1]]) if len(amber_idx) else red_p * 0.6
amber_p, red_p = round(min(amber_p, red_p - 0.01), 3), round(red_p, 3)
print(f"cut-offs from validation: amber_p={amber_p} red_p={red_p}  (put these in Admin > Settings)")

try:
    import matplotlib.pyplot as plt

    from sklearn.calibration import calibration_curve

    frac, mean = calibration_curve(test["y"], p_test, n_bins=10, strategy="quantile")
    plt.plot(mean, frac, "o-"); plt.plot([0, 1], [0, 1], "--", c="grey")
    plt.xlabel("predicted"); plt.ylabel("observed"); plt.title("Reliability (test)"); plt.show()
    lgb.plot_importance(model, max_num_features=15, importance_type="gain"); plt.show()
except ImportError:
    pass

## 8. Export (`ml/models/lgbm_v1.txt` + `features.json`)
Commit both, then copy them into `server/models/` before deploying the Space.

In [ ]:
model.save_model(str(MODEL_DIR / "lgbm_v1.txt"), num_iteration=model.best_iteration)
meta = {
    "version": "lgbm_v1", "features": features,
    "label": f"precipitation(t+1..t+3) >= {LABEL_MM:g} mm (Open-Meteo Archive)", "label_mm": LABEL_MM,
    "forecast_model": FCST_MODEL if USE_FORECAST_FEATURES else None,  # backend fetches this model live
    "train": [START if not USE_FORECAST_FEATURES else FCST_START, TRAIN_END], "valid_end": VALID_END,
    "test": [VALID_END, END], "zones": [z[0] for z in ZONES],
    "metrics": {"valid": m_valid, "test": m_test}, "cutoffs": {"amber_p": amber_p, "red_p": red_p},
    "trained_at": datetime.now(timezone.utc).isoformat(),
}
(MODEL_DIR / "features.json").write_text(json.dumps(meta, indent=2, default=float))
print(json.dumps(meta["metrics"], indent=2, default=float))